In [3]:
!unzip images.zip

Archive:  images.zip
   creating: images/
  inflating: __MACOSX/._images       
  inflating: images/page_2.png       
  inflating: __MACOSX/images/._page_2.png  
  inflating: images/page_3.png       
  inflating: __MACOSX/images/._page_3.png  
  inflating: images/page_19.json     
  inflating: __MACOSX/images/._page_19.json  
  inflating: images/page_1.png       
  inflating: __MACOSX/images/._page_1.png  
  inflating: images/page_35.json     
  inflating: __MACOSX/images/._page_35.json  
  inflating: images/page_44.png      
  inflating: __MACOSX/images/._page_44.png  
  inflating: images/page_22.json     
  inflating: __MACOSX/images/._page_22.json  
  inflating: images/page_40.png      
  inflating: __MACOSX/images/._page_40.png  
  inflating: images/page_4.png       
  inflating: __MACOSX/images/._page_4.png  
  inflating: images/page_34.json     
  inflating: __MACOSX/images/._page_34.json  
  inflating: images/page_18.json     
  inflating: __MACOSX/images/._page_18.json  
  infl

In [5]:
%pip install -U 'git+https://github.com/facebookresearch/detectron2.git@ff53992b1985b63bd3262b5a36167098e3dada02'

  Cloning https://github.com/facebookresearch/detectron2.git (to revision ff53992b1985b63bd3262b5a36167098e3dada02) to /tmp/pip-req-build-cjowaks6
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-cjowaks6
  Running command git rev-parse -q --verify 'sha^ff53992b1985b63bd3262b5a36167098e3dada02'
  Running command git fetch -q https://github.com/facebookresearch/detectron2.git ff53992b1985b63bd3262b5a36167098e3dada02
  Running command git checkout -q ff53992b1985b63bd3262b5a36167098e3dada02
  Resolved https://github.com/facebookresearch/detectron2.git to commit ff53992b1985b63bd3262b5a36167098e3dada02
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.2/79.2 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 12.3 MB/s eta 

In [3]:
#!/usr/bin/env python3

import os
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.data.datasets import register_coco_instances
import argparse

def setup_config(train_dataset_name: str,
                train_json_annot_path: str,
                train_image_path: str,
                val_dataset_name: str = None,
                val_json_annot_path: str = None,
                val_image_path: str = None,
                num_classes: int = 1,
                device: str = "cuda",
                output_dir: str = "./output"):
    """Setup Detectron2 configuration for training."""
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
    cfg.DATASETS.TRAIN = (train_dataset_name,)
    cfg.DATASETS.TEST = (val_dataset_name,) if val_dataset_name else ()

    # Register datasets
    register_coco_instances(train_dataset_name, {}, train_json_annot_path, train_image_path)
    if val_dataset_name:
        register_coco_instances(val_dataset_name, {}, val_json_annot_path, val_image_path)

    # Model configuration
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml")
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = num_classes
    cfg.MODEL.DEVICE = device

    # Training configuration
    cfg.SOLVER.IMS_PER_BATCH = 2
    cfg.SOLVER.BASE_LR = 0.00025
    cfg.SOLVER.MAX_ITER = 1000
    cfg.SOLVER.STEPS = []  # No learning rate decay

    # Output directory
    os.makedirs(output_dir, exist_ok=True)
    cfg.OUTPUT_DIR = output_dir

    return cfg

def train():
    # Setup configuration
    cfg = setup_config(
        train_dataset_name="figures_train",
        train_json_annot_path="annotations.json",
        train_image_path="images/",
        # val_dataset_name="figures_val" if args.val_json else None,
        # val_json_annot_path=args.val_json,
        # val_image_path=args.val_images,
        num_classes=1,
        device="cuda",
        output_dir="output"
    )

    # Train the model
    trainer = DefaultTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()

In [4]:
train()

[12/29 16:20:56 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
model_final_280758.pkl: 167MB [00:01, 144MB/s]                           
roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, weight}


[12/29 16:20:57 d2.engine.train_loop]: Starting training from iteration 0


/usr/local/lib/python3.10/dist-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[12/29 16:21:10 d2.utils.events]:  eta: 0:06:12  iter: 19  total_loss: 0.6011  loss_cls: 0.4642  loss_box_reg: 0.09986  loss_rpn_cls: 0.02745  loss_rpn_loc: 0.006574    time: 0.3805  last_time: 0.3921  data_time: 0.0343  last_data_time: 0.0122   lr: 4.9953e-06  max_mem: 2110M
[12/29 16:21:22 d2.utils.events]:  eta: 0:06:12  iter: 39  total_loss: 0.5425  loss_cls: 0.396  loss_box_reg: 0.1048  loss_rpn_cls: 0.02396  loss_rpn_loc: 0.006584    time: 0.3920  last_time: 0.4216  data_time: 0.0091  last_data_time: 0.0117   lr: 9.9902e-06  max_mem: 2110M
[12/29 16:21:30 d2.utils.events]:  eta: 0:06:11  iter: 59  total_loss: 0.4428  loss_cls: 0.2962  loss_box_reg: 0.105  loss_rpn_cls: 0.02604  loss_rpn_loc: 0.007853    time: 0.4011  last_time: 0.3373  data_time: 0.0115  last_data_time: 0.0052   lr: 1.4985e-05  max_mem: 2110M
[12/29 16:21:38 d2.utils.events]:  eta: 0:06:03  iter: 79  total_loss: 0.3597  loss_cls: 0.2223  loss_box_reg: 0.1206  loss_rpn_cls: 0.01407  loss_rpn_loc: 0.007535    time: